## Imports


In [23]:
import copy
import logging
import os
from pathlib import Path
from typing import Any, Dict, List, Optional

import open_clip
import wandb

import hydra
import omegaconf
import pytorch_lightning as pl
import torch
from hydra import compose, initialize
from hydra.utils import instantiate
from lightning.pytorch import Callback
from omegaconf import DictConfig, ListConfig, OmegaConf
from torch.nn.utils import parameters_to_vector, vector_to_parameters

from nn_core.callbacks import NNTemplateCore
from nn_core.common import PROJECT_ROOT
from nn_core.common.utils import enforce_tags, seed_index_everything
from nn_core.model_logging import NNLogger
from nn_core.serialization import NNCheckpointIO

# Force the execution of __init__.py if this file is executed directly.
import mass  # noqa
from mass.data.datasets.registry import get_dataset
from mass.modules.encoder import ClassificationHead, ImageEncoder
from mass.modules.projection_router import ProjectionRouter
from mass.modules.nn_router import NNRouter
from mass.modules.heads import get_classification_head
from mass.modules.router import AbstractRouter
from mass.utils.io_utils import load_model_from_disk
from mass.utils.plots import plot_interactive_radar_chart
from mass.utils.utils import (
    compute_task_dict, 
    apply_dict_to_model,
    build_callbacks,
    get_finetuning_accuracies,
    add_normalized_accuracy,
    compute_avg_accuracy,
    print_memory,
    get_routing_weights,
    svd_key_from_layer
)
from mass.task_vectors.task_singular_vectors import *
import json
import os

pylogger = logging.getLogger(__name__)

torch.set_float32_matmul_precision("high")

In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import hydra
from hydra import initialize, compose
from typing import Dict, List

hydra.core.global_hydra.GlobalHydra.instance().clear()
initialize(version_base=None, config_path=str("../conf"), job_name="debug_mnist")
cfg = compose(config_name="static_merging")

In [26]:
# upperbound accuracies, used for logging the normalized accuracy
finetuned_accuracies: Dict[str, float] = get_finetuning_accuracies(
    cfg.misc.finetuned_accuracy_path
)

# only has vision encoder, no text transformer
zeroshot_encoder: ImageEncoder = load_model_from_disk(
    cfg.misc.pretrained_checkpoint, model_name=cfg.nn.module.encoder.model_name
)

finetuned_name = (
    lambda name: Path(cfg.misc.ckpt_path) / f"{name}Val" / "nonlinear_finetuned.pt"
)
finetuned_models = {
    dataset: load_model_from_disk(
        finetuned_name(dataset), model_name=cfg.nn.module.encoder.model_name
    ).state_dict()
    for dataset in cfg.benchmark.datasets
}


2025-09-03 11:38:37 INFO     Loading model from disk                                         ]8;id=476089;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/utils/io_utils.py\mass.utils.io_utils]8;;\:]8;id=726290;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/utils/io_utils.py#79\79]8;;\
                             /media/donato/Extra-storage/Code/model-merging/resources/checkp                       
                             oints//ViT-B-32/MNISTVal/nonlinear_zeroshot.pt                                        

2025-09-03 11:38:38 INFO     Loading ViT-B-32 pre-trained weights.                          ]8;id=184536;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/modules/encoder.py\mass.modules.encoder]8;;\:]8;id=565679;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/modules/encoder.py#23\23]8;;\

                    INFO     Loading pretrained ViT-B-32 from OpenAI.                                       ]8;id=41824;file:///media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/open_clip/factory.py\root]8;;\:]8;id=989972;file:///media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/open_clip/factory.py#82\82]8;;\

2025-09-03 11:38:41 INFO     Removing text transformer from the model.                      ]8;id=2891;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/modules/encoder.py\mass.modules.encoder]8;;\:]8;id=39684;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/modules/encoder.py#39\39]8;;\

                    INFO     Loading model from disk                                         ]8;id=122709;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/utils/io_utils.py\mass.utils.io_utils]8;;\:]8;id=414770;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/utils/io_utils.py#79\79]8;;\
                             /media/donato/Extra-storage/Code/model-merging/resources/checkp                       
                             oints/ViT-B-32/SUN397Val/nonlinear_finetuned.pt                                       

...

2025-09-03 11:39:40 INFO     Removing text transformer from the model.                      ]8;id=936750;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/modules/encoder.py\mass.modules.encoder]8;;\:]8;id=283191;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/modules/encoder.py#39\39]8;;\

                    INFO     Loading model from disk                                         ]8;id=468855;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/utils/io_utils.py\mass.utils.io_utils]8;;\:]8;id=948211;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/utils/io_utils.py#79\79]8;;\
                             /media/donato/Extra-storage/Code/model-merging/resources/checkp                       
                             oints/ViT-B-32/RenderedSST2Val/nonlinear_finetuned.pt                                 

2025-09-03 11:39:41 INFO     Loading ViT-B-32 pre-trained weights.                          ]8;id=731736;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/modules/encoder.py\mass.modules.encoder]8;;\:]8;id=77742;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/modules/encoder.py#23\23]8;;\

                    INFO     Loading pretrained ViT-B-32 from OpenAI.                                       ]8;id=371071;file:///media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/open_clip/factory.py\root]8;;\:]8;id=336183;file:///media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/open_clip/factory.py#82\82]8;;\

2025-09-03 11:39:43 INFO     Removing text transformer from the model.                      ]8;id=334080;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/modules/encoder.py\mass.modules.encoder]8;;\:]8;id=1675;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/modules/encoder.py#39\39]8;;\

In [ ]:
for dataset_name in cfg.benchmark.datasets:
    pylogger.info(f"Loading dataset: {dataset_name}")

    dataset_cfg = OmegaConf.load(
    PROJECT_ROOT / "conf" / "dataset" / f"{dataset_name}.yaml"
    )

    dataset = instantiate(
        dataset_cfg, preprocess_fn=zeroshot_encoder.val_preprocess
    )